# La même série, deux histoires · *The same series, two stories*

Notebook compagnon du chapitre **18. Atelier données : découvrir FRED, l'entrepôt de la Réserve fédérale** — [lire l'article](https://nmlab.io/ressources/atelier-donnees-decouvrir-fred).
Companion notebook to chapter **18. Data Workshop: Discovering FRED** — [read the article](https://nmlab.io/en/ressources/data-workshop-discovering-fred).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


# Indice des prix à la consommation, en direct depuis FRED / CPI, live from FRED
cpi = nm.load_fred("CPIAUCSL")

# « Variation sur un an » : la transformation qui change un niveau en inflation.
# "Change from year ago": the transformation that turns a level into inflation.
inflation = (cpi / cpi.shift(12) - 1) * 100

cpi, inflation = cpi.loc["1995":], inflation.loc["1995":]
print(f"Dernier point / latest: {inflation.index[-1]:%Y-%m} \u2192 {inflation.iloc[-1]:.1f} %")


# ── Libellés bilingues / bilingual labels ────────────────────────────────────
MOIS = ["janvier", "février", "mars", "avril", "mai", "juin", "juillet",
        "août", "septembre", "octobre", "novembre", "décembre"]
latest, when = inflation.iloc[-1], inflation.index[-1]
latest_fr = f"{latest:.1f}".replace(".", ",")   # 3.5 → « 3,5 » pour la note française

L = {
    "fr": dict(
        title="La même série, deux histoires",
        sub="Un menu déroulant transforme un niveau en taux d'inflation",
        y1="« Niveau »", y2="« Variation sur un an », %",
        lab1="l'indice des prix (IPC)", target="cible 2 %",
        note="Le même indice des prix, vu comme « Niveau » (en haut) puis comme « Variation sur un an » (en bas) : c'est\n"
             f"ainsi qu'on lit l'inflation, à {latest_fr} % en {MOIS[when.month - 1]} {when.year} (dernier point). "
             "Source : BLS via FRED (CPIAUCSL)."),
    "en": dict(
        title="The same series, two stories",
        sub="One dropdown turns a level into an inflation rate",
        y1="« Level »", y2="« Change from year ago », %",
        lab1="the price index (CPI)", target="2% target",
        note="The same price index, seen as « Level » (top) then as « Change from year ago » (bottom): that is how you\n"
             f"read inflation, at {latest:.1f}% in {when:%B %Y} (latest point). Source: BLS via FRED (CPIAUCSL)."),
}[LANG]

# ── Figure : deux panneaux (niveau en haut, variation en bas) ─────────────────
import matplotlib.dates as mdates

fig = nm.figure(height_px=1140)
ax1 = fig.add_axes([0.075, 0.553, 0.907, 0.225])
ax2 = fig.add_axes([0.075, 0.140, 0.907, 0.307])

ax1.plot(cpi.index, cpi, color=nm.COLORS["blue"], linewidth=3.2)
ax1.set_ylabel(L["y1"])
ax1.set_yticks([200, 300])
ax1.text(0.065, 0.80, L["lab1"], transform=ax1.transAxes, fontsize=21.5,
         color=nm.COLORS["muted"])
ax1.tick_params(labelbottom=False)

ax2.plot(inflation.index, inflation, color=nm.COLORS["rose"], linewidth=3.2)
ax2.axhline(2, color=nm.COLORS["amber"], linestyle=(0, (6, 4)), linewidth=2.6)
ax2.axhline(0, color=nm.COLORS["muted"], linewidth=1.6, alpha=0.9)
ax2.set_ylabel(L["y2"])
ax2.set_yticks(range(-2, 9, 2))
ax2.text(0.065, 0.80, L["target"], transform=ax2.transAxes, fontsize=21.5,
         fontweight="bold", color=nm.COLORS["amber"])

for ax in (ax1, ax2):
    ax.set_xlim(cpi.index[0], cpi.index[-1])
    ax.margins(x=0.012)
    ax.xaxis.set_major_locator(mdates.YearLocator(5))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

nm.header(fig, L["title"], L["sub"])
nm.footer(fig, L["note"])